# 1. Librerias y conjunto de datos

In [125]:
import os
import requests
import pickle
from joblib import dump
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import missingno as msno
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors 

In [104]:
df = pd.read_csv(r"/workspaces/steven10015-intro-ml/data/raw/adult-census-income.csv")
df.head(10)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K
7,74,State-gov,88638,Doctorate,16,Never-married,Prof-specialty,Other-relative,White,Female,0,3683,20,United-States,>50K
8,68,Federal-gov,422013,HS-grad,9,Divorced,Prof-specialty,Not-in-family,White,Female,0,3683,40,United-States,<=50K
9,41,Private,70037,Some-college,10,Never-married,Craft-repair,Unmarried,White,Male,0,3004,60,?,>50K


In [105]:
df.shape

(32561, 15)

# 2. Cribado

## 2.1 Valores únicos

In [106]:
df.nunique()

age                  73
workclass             9
fnlwgt            21648
education            16
education.num        16
marital.status        7
occupation           15
relationship          6
race                  5
sex                   2
capital.gain        119
capital.loss         92
hours.per.week       94
native.country       42
income                2
dtype: int64

## 2.2 Filas duplicadas

In [107]:
duplicados = df.duplicated().sum()
print(f"Hay {duplicados} filas duplicadas en el DataFrame.")

Hay 24 filas duplicadas en el DataFrame.


In [108]:
df.drop_duplicates(inplace=True)

## 2.3 Columnas duplicadas

In [109]:
df.T.duplicated().sum()

np.int64(0)

## 2.4 Valores Nan

In [110]:
# Ver cuántos nulos hay por columna
df.replace('?', np.nan, inplace=True)
print(df.isnull().sum())

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     582
income               0
dtype: int64


In [111]:
df.isnull().mean()*100

age               0.000000
workclass         5.642807
fnlwgt            0.000000
education         0.000000
education.num     0.000000
marital.status    0.000000
occupation        5.664321
relationship      0.000000
race              0.000000
sex               0.000000
capital.gain      0.000000
capital.loss      0.000000
hours.per.week    0.000000
native.country    1.788733
income            0.000000
dtype: float64

In [112]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

In [113]:
df.shape

(30139, 15)

#### A pesar de ser porcentajos tan pequeños no se considera la imputación por el tipo de variables que son, imputar un país u ocupación puede introducir mucho ruido en los datos. La cantidad de filas que se perderían es mínima, no habría un gran impacto.

## 2.5 Columnas redundantes

In [114]:
# Es redundante, ya se tiene a education-num 
df.drop(columns=['education'], inplace=True)

In [115]:
# Columna 'fnlwgt' tiene 21647 valores únicos, lo que sugiere que es un identificador único para cada fila.
# Por lo tanto, no aporta información útil para el análisis y puede ser eliminada.
df.drop(columns=['fnlwgt'], inplace=True)

In [116]:
# Redundante con marital-status
df.drop(columns=['relationship'], inplace=True)

## 2.6 Variables categóricas

In [117]:
# 1. Transformar el Target (income) a 0 y 1
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

# 2. Identificar variables categóricas para transformar
# Excluimos las numéricas y el target
categorical_cols = df.select_dtypes(include=['object']).columns

# Aplicamos One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Nuevas dimensiones tras One-Hot Encoding: {df_final.shape}")

Nuevas dimensiones tras One-Hot Encoding: (30139, 76)


## 2.7 Normalización

In [122]:
# Cambiamos puntos por guiones bajos en todo el DataFrame
df_final.columns = [col.replace('.', '_') for col in df_final.columns]

# Ahora verificamos que existan
print("Columnas actuales:", df_final.columns[:10]) # Solo las primeras 10

Columnas actuales: Index(['age', 'education_num', 'capital_gain', 'capital_loss',
       'hours_per_week', 'income', 'workclass_Local-gov', 'workclass_Private',
       'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc'],
      dtype='object')


In [123]:
# 1. Definimos nuestras variables de entrada (X) y el objetivo (y)
X = df_final.drop('income', axis=1)
y = df_final['income']

# 2. Identificamos las columnas numéricas originales que necesitan escalado
# (Las columnas que NO son resultado del One-Hot Encoding)
numeric_cols = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

# 3. Aplicamos el Scaler
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# 4. Dividimos el set de datos en entrenamiento y prueba (80/20)
# Esto es fundamental para evaluar el modelo después.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("¡Normalización y división de datos completada!")
print(f"Set de entrenamiento: {X_train.shape}")
print(f"Set de prueba: {X_test.shape}")

¡Normalización y división de datos completada!
Set de entrenamiento: (24111, 75)
Set de prueba: (6028, 75)


# 3. Planteamiento de problema

En este proyecto, no estamos construyendo el típico recomendador de películas o productos. Lo que buscamos es un sistema de recomendación de movilidad socioeconómica. La idea es que, una vez que el modelo puede predecir quién gana menos de 50k, el sistema actúe como un consultor estratégico que analiza qué pequeños cambios en la vida de esa persona podrían abrirle la puerta a un nivel de ingresos superior.

¿Qué es lo que realmente vamos a recomendar?  
El sistema sugerirá ajustes accionables en el perfil del usuario. No se trata de decir "tienes que nacer en otro país" o "tienes que ser más joven", porque eso es imposible de cambiar. El foco estará en variables que la persona sí puede gestionar, como elevar su nivel educativo, cambiar el tipo de industria en la que trabaja o ajustar su carga horaria semanal. Básicamente, el sistema recomienda el "camino más corto" para cambiar la predicción de ingresos de negativa a positiva.

¿A quién va dirigido? (El Usuario)  
Nuestro usuario es cualquier adulto que, según los datos del censo, se encuentra actualmente por debajo del umbral de los 50,000 USD anuales. Es alguien que busca entender qué factores de su perfil están pesando más en su realidad económica y qué palancas podría mover para mejorar su situación según los patrones históricos que el modelo ha detectado.

¿Cómo definimos el perfil del usuario?  
El perfil no es solo un número, es una combinación de tres dimensiones:

Su formación: Representada principalmente por sus años de estudio.

Su situación laboral: Qué hace, en qué sector trabaja y cuánto tiempo le dedica a la semana.

Su entorno: Aquí entran los factores sociales como el género, la raza o el estado civil. Aunque el usuario no puede cambiar estos rasgos, el sistema los necesita para entender desde qué posición social está partiendo y cómo esos factores influyen en las probabilidades que el modelo calcula.

# 4. Sistema de recomendación

In [130]:
# 1. Identificamos en el set de entrenamiento quiénes son los "exitosos"
# Usamos y_train porque son las etiquetas reales de esos datos
indices_exito_train = y_train[y_train == 1].index
X_exito_train = X_train.loc[indices_exito_train]

# 2. Entrenamos el buscador de vecinos solo con esos perfiles de éxito
# X_exito_train ya está normalizado (lo hiciste en el paso 3 de la directriz 2)
recommender = NearestNeighbors(n_neighbors=1, metric='cosine')
recommender.fit(X_exito_train)

# 3. Elegimos a un usuario de X_test que el modelo debería ayudar
# Por ejemplo, el primero que gana <= 50K
indices_bajos_ingresos_test = y_test[y_test == 0].index
idx_usuario_test = indices_bajos_ingresos_test[0]
usuario_vector = X_test.loc[[idx_usuario_test]]

# 4. Buscamos al "gemelo de éxito" más cercano
distancia, indice_en_matriz = recommender.kneighbors(usuario_vector)

# 5. Obtenemos el índice original del vecino para poder comparar
idx_vecino_exito = X_exito_train.index[indice_en_matriz[0][0]]

print(f"Análisis completado para el usuario: {idx_usuario_test}")
print(f"Su referente de éxito es el usuario: {idx_vecino_exito}")

Análisis completado para el usuario: 15756
Su referente de éxito es el usuario: 22834


In [132]:
# 1. Aseguramos que el df original tenga los mismos nombres que el df_final
df.columns = [col.replace('.', '_') for col in df.columns]

# 2. Extraemos los datos usando los índices que encontramos
actual = df.loc[15756]
sugerido = df.loc[22834]

# 3. Construimos la comparativa (ahora con guiones bajos)
comparativa = {
    'Variable': ['Años Educación', 'Horas Semanales', 'Ocupación', 'Sector'],
    'Usuario Actual': [actual['education_num'], actual['hours_per_week'], actual['occupation'], actual['workclass']],
    'Perfil de Éxito': [sugerido['education_num'], sugerido['hours_per_week'], sugerido['occupation'], sugerido['workclass']]
}

df_reporte = pd.DataFrame(comparativa)
print("--- Comparativa de Perfiles ---")
print(df_reporte)

--- Comparativa de Perfiles ---
          Variable   Usuario Actual  Perfil de Éxito
0   Años Educación                7                7
1  Horas Semanales               45               50
2        Ocupación  Exec-managerial  Exec-managerial
3           Sector          Private          Private


In [135]:
# apartado social 
print(f"Análisis de contexto social:")
print(f"Usuario Actual: Sex: {actual['sex']}, Race: {actual['race']}")
print(f"Perfil de Éxito: Sex: {sugerido['sex']}, Race: {sugerido['race']}")

Análisis de contexto social:
Usuario Actual: Sex: Male, Race: White
Perfil de Éxito: Sex: Male, Race: White


#### Siendo un hombre blanco en el sector privado con tu misma educación, la diferencia entre ganar más o menos de 50k reside en esas 5 horas extra de jornada semanal.

# 5. Caso simulado

## 5.1 Definimos el perfil

In [ ]:
nuevo_usuario_raw = {
    'age': 25,
    'workclass': 'Private',
    'education_num': 9,
    'marital_status': 'Never-married',
    'occupation': 'Other-service',
    'race': 'White',
    'sex': 'Male',
    'capital_gain': 0,
    'capital_loss': 0,
    'hours_per_week': 20,
    'native_country': 'United-States'
}

# Al hacer esto, garantizamos que coincida con X_train
usuario_simulado = pd.DataFrame([nuevo_usuario_raw])

# El reindex es lo que nos salva la vida:
# Si falta alguna columna que el One-Hot creó (como 'workclass_Self-emp'), 
# reindex le pone un 0 automáticamente.
usuario_simulado_dummies = pd.get_dummies(usuario_simulado)
usuario_simulado_final = usuario_simulado_dummies.reindex(columns=X_train.columns, fill_value=0)

usuario_simulado_final[numeric_cols] = scaler.transform(usuario_simulado_final[numeric_cols])

## 5.2 Test final

In [139]:
# 1. Buscamos al "gemelo de éxito" más cercano
distancia, indice_en_matriz = recommender.kneighbors(usuario_simulado_final)
idx_referente = X_exito_train.index[indice_en_matriz[0][0]]
perfil_referente = df.loc[idx_referente]

print(f"--- RESULTADOS PARA EL USUARIO SIMULADO ---")
print(f"Perfil inicial: 25 años, {nuevo_usuario_raw['education_num']} años estudio, {nuevo_usuario_raw['hours_per_week']}h/semana.")
print(f"Referente de éxito encontrado: ID {idx_referente}")
print("-" * 45)

print("TRAYECTORIA RECOMENDADA PARA SUPERAR LOS 50K:")

# Comparación de Educación
if perfil_referente['education_num'] > nuevo_usuario_raw['education_num']:
    print(f"-> EDUCACIÓN: Subir de {nuevo_usuario_raw['education_num']} a {perfil_referente['education_num']} años de estudio.")
else:
    print(f"-> EDUCACIÓN: Tu nivel educativo actual es similar al de perfiles exitosos.")

# Comparación de Horas
if perfil_referente['hours_per_week'] > nuevo_usuario_raw['hours_per_week']:
    print(f"-> JORNADA: Aumentar de {nuevo_usuario_raw['hours_per_week']}h a {perfil_referente['hours_per_week']}h semanales.")

# Comparación de Ocupación
if perfil_referente['occupation'] != nuevo_usuario_raw['occupation']:
    print(f"-> OCUPACIÓN: Evalúa moverte hacia roles de tipo: {perfil_referente['occupation']}.")

--- RESULTADOS PARA EL USUARIO SIMULADO ---
Perfil inicial: 25 años, 9 años estudio, 20h/semana.
Referente de éxito encontrado: ID 22722
---------------------------------------------
TRAYECTORIA RECOMENDADA PARA SUPERAR LOS 50K:
-> EDUCACIÓN: Subir de 9 a 10 años de estudio.
-> JORNADA: Aumentar de 20h a 32h semanales.


Transformación de Datos: El proceso de limpieza y codificación One-Hot permitió convertir variables categóricas complejas en un espacio vectorial de 75 dimensiones, facilitando el análisis matemático de perfiles sociales.  

Modelo de Recomendación: Al implementar un enfoque de Filtrado Basado en Contenido con la métrica de Similitud de Coseno, logramos que el sistema identifique "gemelos de éxito". Esto garantiza que las recomendaciones no sean genéricas, sino personalizadas según el contexto de cada individuo.  

Impacto de las Variables: Las pruebas simuladas confirman que pequeñas variaciones en la formación académica y la carga horaria son los catalizadores más eficientes para mejorar el nivel de ingresos, validando la importancia de las variables accionables identificadas en la etapa de exploración.

# 6. Guardar

In [140]:
folder_path = '/workspaces/steven10015-intro-ml/models'

# 2. Creamos la carpeta si no existe
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# 3. Guardamos el recomendador k-NN
with open(os.path.join(folder_path, 'recommender_model.pkl'), 'wb') as f:
    pickle.dump(recommender, f)

# 4. Guardamos el Scaler (Crucial para procesar nuevos usuarios)
with open(os.path.join(folder_path, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# 5. Guardamos el DataFrame de éxito (Lo necesita el k-NN para comparar)
with open(os.path.join(folder_path, 'exito_profiles.pkl'), 'wb') as f:
    pickle.dump(X_exito_train, f)

# 6. Guardamos la lista de columnas para asegurar consistencia
with open(os.path.join(folder_path, 'model_columns.pkl'), 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

print(f"✅ Sistema de recomendación guardado en: {folder_path}")

✅ Sistema de recomendación guardado en: /workspaces/steven10015-intro-ml/models
